# Create edgotype profiles

In this step, we make one edgotype profile per gene of interactors by alleles. Later, we will explore metrics that summarize overall perturbation for each allele (quantitative score like imaging / abundance)

# Automated

In [1]:
import polars as pl

In [2]:
inputs_dir = "../1_inputs/sqY2H"
outputs_dir = "../3_outputs/sqY2H"

In [3]:
import os
import shutil

# Remove all files in subfolders edgotyping_profiles
profile_dirs = [
    # f"{outputs_dir}/edgotype_profiles/manual",
    f"{outputs_dir}/edgotype_profiles/auto_norm", 
    f"{outputs_dir}/edgotype_profiles/auto_summed",
    f"{outputs_dir}/edgotype_profiles/auto_lwh1"
]

for profile_dir in profile_dirs:
    if os.path.exists(profile_dir):
        # Remove all files in the directory
        for filename in os.listdir(profile_dir):
            file_path = os.path.join(profile_dir, filename)
            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.unlink(file_path)
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
            except Exception as e:
                print(f'Failed to delete {file_path}. Reason: {e}')
        print(f"Cleared all files from {profile_dir}")
    else:
        print(f"Directory {profile_dir} does not exist")

Cleared all files from ../3_outputs/sqY2H/edgotype_profiles/auto_norm
Cleared all files from ../3_outputs/sqY2H/edgotype_profiles/auto_summed
Cleared all files from ../3_outputs/sqY2H/edgotype_profiles/auto_lwh1


In [4]:
scores = pl.read_csv(f"{outputs_dir}/4_automated_normalized_consensus_scores.csv")
scores = scores.drop_nulls(subset=["summed_consensus_score"])

# keep only scores where there is at least one matched reference-variant set
orfs_with_alleles = scores.filter(pl.col("db_mut_id") != 0).select("db_orf_id").unique().to_series().to_list()
orfs = scores.filter(pl.col("db_mut_id") == 0).select("db_orf_id").unique().to_series().to_list()

complete_orf_pairs = [i for i in orfs if i in orfs_with_alleles]
scores = scores.filter(pl.col("db_orf_id").is_in(complete_orf_pairs))
genes = scores.select("db_orf_id").to_series().unique().to_list()


scores_for_export = scores.select([
    "db_orf_id", "ad_orf_id", "db_mut_id", "symbol", "ad_symbol", "aa_change", "nt_change", "ensembl_gene_id", "clinvar_clnsig_clean",
    "summed_consensus_score", "summed_consensus_score_normalized"
])

scores_for_export.write_csv(f"{outputs_dir}/4_automated_normalized_consensus_scores_filtered_wt_variant_pairs.csv")
scores_for_export

db_orf_id,ad_orf_id,db_mut_id,symbol,ad_symbol,aa_change,nt_change,ensembl_gene_id,clinvar_clnsig_clean,summed_consensus_score,summed_consensus_score_normalized
i64,i64,i64,str,str,str,str,str,str,f64,f64
2651,12260,0,"""CLN6""","""STX1A""",null,null,"""ENSG00000128973""",null,634.149032,0.211383
52920,71163,2687,"""LITAF""","""KCNIP1""","""Pro34Thr""","""100C>A""","""ENSG00000189067""","""4_VUS""",1290.260496,0.430087
5066,152,201832,"""BAG3""","""TRIM27""","""Pro380Ser""","""1138C>T""","""ENSG00000151929""","""2_Benign""",435.35981,0.14512
4062,4946,201423,"""PLP1""","""PTPRN""","""Gly74Arg""","""220G>C""","""ENSG00000123560""",null,86.33072,0.028777
3535,12303,0,"""EMD""","""GORAB""",null,null,"""ENSG00000102119""",null,1414.35952,0.471453
…,…,…,…,…,…,…,…,…,…,…
3980,53626,201376,"""GRN""","""LCE4A""","""Thr182Met""","""545C>T""","""ENSG00000030582""","""2_Benign""",986.960477,0.328987
52920,1235,2678,"""LITAF""","""HPCAL4""","""Ala111Gly""","""332C>G""","""ENSG00000189067""","""1_Pathogenic""",1224.815397,0.408272
13796,8297,218140,"""HBD""","""HBZ""","""Gly17Arg""","""49G>C""","""ENSG00000223609""","""5_Others""",530.595443,0.176865


In [5]:
scores_for_export.select(pl.col('symbol')).n_unique()

73

In [10]:
scores

ad_orf_id,db_orf_id,db_mut_id,retest_batch,n_conditions,summed_consensus_score,lwh1_consensus_score,summed_consensus_score_normalized,lwh1_consensus_score_normalized,wt_summed_score,wt_summed_score_normalized,wt_lwh1_score,wt_lwh1_score_normalized,condition_name,assay_id,ppi_group,seq_ad,seq_db,ad_symbol,symbol,aa_change,nt_change,ensembl_gene_id,collection,clinvar_clnsig_clean,gnomad_af,StarStatus,allele_0
i64,i64,i64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,str,i64,str,i64,i64,str,str,str,str,str,str,str,f64,f64,str
12260,2651,0,"""VUSAPWT1B1""",4,634.149032,327.398046,0.211383,0.436531,634.149032,0.211383,327.398046,0.436531,"""LWA""",1,"""12260_2651_0_VUSAPWT1B1""",1,1,"""STX1A""","""CLN6""",null,null,"""ENSG00000128973""",null,null,null,null,"""CLN6"""
71163,52920,2687,"""VUSAPWT2B1""",4,1290.260496,554.659173,0.430087,0.739546,1247.363108,0.415788,523.020282,0.69736,"""LWH25""",2,"""71163_52920_2687_VUSAPWT2B1""",1,1,"""KCNIP1""","""LITAF""","""Pro34Thr""","""100C>A""","""ENSG00000189067""","""Edgotyping3""","""4_VUS""",null,2.0,"""LITAF_Pro34Thr"""
152,5066,201832,"""VUSAPWT1B2""",4,435.35981,235.246735,0.14512,0.313662,390.37844,0.130126,null,null,"""LWA""",1,"""152_5066_201832_VUSAPWT1B2""",1,1,"""TRIM27""","""BAG3""","""Pro380Ser""","""1138C>T""","""ENSG00000151929""","""CEGS2""","""2_Benign""",0.000499,2.0,"""BAG3_Pro380Ser"""
4946,4062,201423,"""VUSAPWT1B2""",4,86.33072,21.807517,0.028777,0.029077,1111.980743,0.37066,365.038571,0.486718,"""LWA""",1,"""4946_4062_201423_VUSAPWT1B2""",1,1,"""PTPRN""","""PLP1""","""Gly74Arg""","""220G>C""","""ENSG00000123560""","""CEGS2""",null,null,null,"""PLP1_Gly74Arg"""
12303,3535,0,"""VUSAPWT6B1""",4,1414.35952,489.286291,0.471453,0.652382,1414.35952,0.471453,489.286291,0.652382,"""LWA""",6,"""12303_3535_0_VUSAPWT6B1""",1,1,"""GORAB""","""EMD""",null,null,"""ENSG00000102119""",null,null,null,null,"""EMD"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
53626,3980,201376,"""VUSAPWT2B1""",4,986.960477,419.674193,0.328987,0.559566,1033.064727,0.344355,352.26831,0.469691,"""LWA""",2,"""53626_3980_201376_VUSAPWT2B1""",1,1,"""LCE4A""","""GRN""","""Thr182Met""","""545C>T""","""ENSG00000030582""","""CEGS2""","""2_Benign""",0.00056,2.0,"""GRN_Thr182Met"""
1235,52920,2678,"""VUSAPWT2B1""",4,1224.815397,434.543806,0.408272,0.579392,1728.347445,0.576116,479.31309,0.639084,"""LWA""",2,"""1235_52920_2678_VUSAPWT2B1""",1,1,"""HPCAL4""","""LITAF""","""Ala111Gly""","""332C>G""","""ENSG00000189067""","""RC4""","""1_Pathogenic""",null,2.0,"""LITAF_Ala111Gly"""
8297,13796,218140,"""VUSAPWT1B2""",4,530.595443,null,0.176865,null,590.109826,0.196703,null,null,"""LWH1""",1,"""8297_13796_218140_VUSAPWT1B2""",1,1,"""HBZ""","""HBD""","""Gly17Arg""","""49G>C""","""ENSG00000223609""","""CEGS2""","""5_Others""",null,null,"""HBD_Gly17Arg"""


In [6]:
# Create profiles for each profiled gene

profiles = {}
shape_summary = []
for gene in genes:
    gene_scores = scores.filter(pl.col("db_orf_id") == gene)
    duplicates = gene_scores.group_by(["ad_orf_id", "db_mut_id"]).len().filter(pl.col("len") > 1)
    profile = gene_scores.pivot(
                values="summed_consensus_score_normalized",
                index="ad_orf_id",
                on="db_mut_id"
            )
    profiles[gene] = profile

    shape_summary.append({
        "gene": gene,
        "n_interactors": profile.height,
        "n_alleles": profile.width
    })

    profile.write_csv(f"{outputs_dir}/edgotype_profiles/auto_summed/orf_id_{gene}_auto_summed.csv")

shape_summary = pl.DataFrame(shape_summary)

In [7]:
scores_lwh1 = scores.drop_nulls(subset=["lwh1_consensus_score", "wt_lwh1_score"])

orfs_with_alleles = scores_lwh1.filter(pl.col("db_mut_id") != 0).select("db_orf_id").unique().to_series().to_list()
orfs = scores_lwh1.filter(pl.col("db_mut_id") == 0).select("db_orf_id").unique().to_series().to_list()
complete_orf_pairs = [i for i in orfs if i in orfs_with_alleles]
scores_lwh1 = scores_lwh1.filter(pl.col("db_orf_id").is_in(complete_orf_pairs))
genes = scores_lwh1.select("db_orf_id").to_series().unique().to_list()


In [8]:
# Create profiles for each profiled gene

profiles = {}
shape_summary = []
for gene in genes:
    gene_scores = scores_lwh1.filter(pl.col("db_orf_id") == gene)
    duplicates = gene_scores.group_by(["ad_orf_id", "db_mut_id"]).len().filter(pl.col("len") > 1)
    profile = gene_scores.pivot(
                values="lwh1_consensus_score",
                index="ad_orf_id",
                on="db_mut_id"
            )
    profiles[gene] = profile

    shape_summary.append({
        "gene": gene,
        "n_interactors": profile.height,
        "n_alleles": profile.width
    })

    profile.write_csv(f"{outputs_dir}/edgotype_profiles/auto_lwh1/orf_id_{gene}_auto_lwh1.csv")

shape_summary = pl.DataFrame(shape_summary)

In [9]:
# ## Add a perturbation category for each edge to use for the N2H validation

# # Extract WT scores
# wt_scores = scores.filter(pl.col("db_mut_id") == 0).select(["db_orf_id", "ad_orf_id", "normalized_score"])

# # Merge WT scores back onto the main df
# scores_with_wt = scores.join(wt_scores, on=["db_orf_id", "ad_orf_id"], how="left", suffix="_wt")

# # Compute allele/WT ratio
# scores_with_wt = scores_with_wt.with_columns(
#     (pl.col("normalized_score") / pl.col("normalized_score_wt")).alias("ratio")
# )

# # Determine if the allele is perturbed - binary output
# threshold = 0.5
# scores_with_wt = scores_with_wt.with_columns(
#     ((pl.col("db_mut_id") != 0) & (pl.col("ratio") < threshold)).alias("perturbed")
# )

# # Build perturbation_classes expression
# bins = [
#     (1.0, float("inf"), "increased edge"),
#     (0.8, 1.0, "similar"),
#     (0.6, 0.8, "slightly perturbed"),
#     (0.4, 0.6, "perturbed"),
#     (0.2, 0.4, "highly perturbed"),
#     (0.0, 0.2, "very highly perturbed"),
# ]

# perturbation_expr = pl.when(pl.col("db_mut_id") == 0).then(pl.lit("WT"))

# perturbation_expr = perturbation_expr.when(pl.col("normalized_score_wt") == 0.0).then(pl.lit("WT edge lost"))

# perturbation_expr = perturbation_expr.when(pl.col("ratio") == 0.0).then(pl.lit("edge lost"))

# perturbation_expr = perturbation_expr.when(pl.col("ratio") == 1.0).then(pl.lit("WT-like"))

# for lower, upper, label in bins:
#     perturbation_expr = perturbation_expr.when((pl.col("ratio") > lower) & (pl.col("ratio") <= upper)).then(pl.lit(label))


# perturbation_expr = perturbation_expr.otherwise(pl.lit("unclassified"))




# scores_with_wt = scores_with_wt.with_columns(
#     perturbation_expr.alias("perturbation_classes")
# )

# scores_with_perturbed = scores_with_wt.select("*")


# pert2 = (pl.when(pl.col("db_mut_id") == 0).then(pl.lit('WT')))

# pert2 = pert2.when(pl.col("normalized_score_wt") == 0.0).then(pl.lit("WT edge lost"))

# pert2 = pert2.when(pl.col("ratio") == 0.0).then(pl.lit("edge lost"))

# pert2 = pert2.when(pl.col("ratio") == 1.0).then(pl.lit("WT-like"))

# bins_2 = [
#     (float("inf"), 1.0, "increased edge"),
#     (1, 0.7, 'slightly perturbed'),
#     (0.7, 0.3, 'perturbed'),
#     (0.3, 0, 'highly perturbed')
# ]

# for upper, lower, label in bins_2:
#     pert2 = pert2.when(
#         (pl.col("ratio") <= upper) &
#         (pl.col("ratio") >  lower)
#     ).then(pl.lit(label))

# pert2 = pert2.otherwise(pl.lit("unclassified"))

# scores_with_perturbed = scores_with_wt.with_columns(
#     perturbation_classes_2 = pert2.alias("perturbation_classes_2")
# )



# cols_to_keep = ['ad_orf_id',
#  'db_orf_id',
#  'db_mut_id',
#  'normalized_score',
#  'ad_symbol',
#  'symbol',
#  'clinvar_clnsig_clean',
#  'normalized_score_wt',
#  'ratio',
#  'perturbed',
#  'perturbation_classes',
#  "perturbation_classes_2"]

# scores_with_perturbed = scores_with_perturbed.select(cols_to_keep)

# scores_with_perturbed.write_csv(f"{outputs_dir}/7_edge_perturbation_categorical.csv")